Social Value Lancashire

Importing Libraries Below


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    


Settings Section

In [ ]:
EXCEL_FILE = "SocialValueData.xlsx"
CSV_DIR = "results_csv"
FIG_DIR = "figures"

os.makedirs(CSV_DIR, exist_ok=True)

# The 14 real Lancashire local authorities
CANONICAL_LAS = {
    "Blackburn with Darwen", "Blackpool", "Burnley", "Chorley", "Fylde",
    "Hyndburn", "Lancaster", "Pendle", "Preston", "Ribble Valley",
    "Rossendale", "South Ribble", "West Lancashire", "Wyre",
}

#min years and LAs for forecast
MIN_LAS_FOR_FORECAST = 10
MIN_YEARS_FOR_FORECAST = 5
FORECAST_YEARS_AHEAD = 3

# IMD groups to us as predictors of crime
NON_CRIME_IMD_DOMAINS = ["Income", "Employment", "Health", "Education", "Barriers", "Living"]

CLUSTER_COLORS = ["#2A10BE", "#C715A0", "#1BD415", "#DB7413"]

#condensed list features for the correlation heatmap- too many illegible
HEATMAP_FEATURES = [
    "children_low_income_relative_level",
    "residual_waste_per_household_kg_level",
    "pct_waste_recycled_level",
    "anxiety_high_pct_level",
    "life_satisfaction_low_pct_level",
    "rough_sleeping_single_night_level",
    "post16_18_positive_destination_pct_level",
    "pct_adults_active_raw_level",
    "mean_imd_score",
    "imd_domain_Barriers",
    "imd_domain_Health",
    "imd_domain_Living",
]

HEATMAP_LABELS = {
    "gdhi_level": "GDHI (level)", "gdhi_trend": "GDHI (trend)",
    "children_low_income_relative_level": "Child. low income\n(level)",
    "children_low_income_relative_trend": "Child. low income\n(trend)",
    "residual_waste_per_household_kg_level": "Residual waste\n(level)",
    "residual_waste_per_household_kg_trend": "Residual waste\n(trend)",
    "pct_waste_recycled_level": "% waste\nrecycled (level)",
    "pct_waste_recycled_trend": "% waste\nrecycled (trend)",
    "anxiety_high_pct_level": "Anxiety, high\n(level)",
    "anxiety_high_pct_trend": "Anxiety, high\n(trend)",
    "life_satisfaction_low_pct_level": "Life satisf., low\n(level)",
    "life_satisfaction_low_pct_trend": "Life satisf., low\n(trend)",
    "rough_sleeping_single_night_level": "Rough sleeping,\nsingle night (level)",
    "rough_sleeping_single_night_trend": "Rough sleeping,\nsingle night (trend)",
    "post16_18_positive_destination_pct_level": "Post-16-18 positive\ndestination (level)",
    "post16_18_positive_destination_pct_trend": "Post-16-18 positive\ndestination (trend)",
    "pct_adults_active_raw_level": "% adults active,\nraw (level)",
    "pct_adults_active_raw_trend": "% adults active,\nraw (trend)",
    "mean_imd_score": "IMD (overall)",
    "imd_domain_Barriers": "IMD: Barriers",
    "imd_domain_Health": "IMD: Health",
    "imd_domain_Living": "IMD: Living Env.",
}

#Figure stuff
plt.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
})

Helper Code

In [ ]:
def clean_la_name(name):
    #Cleaning local authority names so consistent
    if pd.isna(name):
        return np.nan
    name = str(name).strip()
    for suffix in [" Borough Council", " City Council", " District Council", " Council", " LA", " District"]:
        if name.endswith(suffix):
            name = name[: -len(suffix)].strip()
    return name


def load_sheet(xls, keyword):
    #find sheet
    matches = [s for s in xls.sheet_names if keyword.lower() in s.lower().replace(" ", "")]
    if not matches:
        raise ValueError(f"No sheet matching '{keyword}' found in the workbook.")
    print(f"  loading sheet '{matches[0]}'")
    return pd.read_excel(xls, sheet_name=matches[0])
